In [ ]:
import numpy as np
import matplotlib.pyplot as plt

#PART A: SIMPLE PERCEPTRON
class Perceptron:
    def __init__(self):
        self.weights = None
        self.bias = 0
    def train(self, X, y, epochs=20):
        self.weights = np.zeros(X.shape[1])
        for _ in range(epochs):
            for i in range(len(X)):
                prediction = 1 if np.dot(X[i], self.weights) + self.bias >= 0 else 0
                error = y[i] - prediction
                self.weights += error * X[i]
                self.bias += error
    def predict(self, X):
        return np.array([1 if np.dot(x, self.weights) + self.bias >= 0 else 0 for x in X])

In [ ]:
#PART B: 2-LAYER NEURAL NETWORK
class TwoLayerNN:
    def __init__(self):
        # Small random weights
        self.W1 = np.random.randn(2, 4) * 0.5
        self.b1 = np.zeros(4)
        self.W2 = np.random.randn(4, 1) * 0.5
        self.b2 = np.zeros(1)
    def sigmoid(self, x):
        return 1 / (1 + np.exp(-x))
    def train(self, X, y, epochs=5000):
        y = y.reshape(-1, 1)
        for _ in range(epochs):
            # Forward pass
            hidden = self.sigmoid(np.dot(X, self.W1) + self.b1)
            output = self.sigmoid(np.dot(hidden, self.W2) + self.b2)
            # Backward pass (simple gradient descent)
            output_error = output - y
            hidden_error = output_error.dot(self.W2.T) * hidden * (1 - hidden)
            # Update weights
            self.W2 -= 0.5 * hidden.T.dot(output_error)
            self.b2 -= 0.5 * np.sum(output_error, axis=0)
            self.W1 -= 0.5 * X.T.dot(hidden_error)
            self.b1 -= 0.5 * np.sum(hidden_error, axis=0)
    def predict(self, X):
        hidden = self.sigmoid(np.dot(X, self.W1) + self.b1)
        output = self.sigmoid(np.dot(hidden, self.W2) + self.b2)
        return (output > 0.5).astype(int).flatten()

In [ ]:
# PART C: SIMPLE KERAS-LIKE IMPLEMENTATION
class SimpleKeras:
    def __init__(self, hidden_neurons):
        self.W1 = np.random.randn(2, hidden_neurons) * 0.5
        self.b1 = np.zeros(hidden_neurons)
        self.W2 = np.random.randn(hidden_neurons, 1) * 0.5
        self.b2 = np.zeros(1)
        self.losses = []
        self.accuracies = []
    def sigmoid(self, x):
        return 1 / (1 + np.exp(-x))
    def fit(self, X, y, epochs=1000):
        y = y.reshape(-1, 1)
        for _ in range(epochs):
            # Forward
            hidden = self.sigmoid(np.dot(X, self.W1) + self.b1)
            output = self.sigmoid(np.dot(hidden, self.W2) + self.b2)
            # Loss
            loss = np.mean((y - output) ** 2)
            self.losses.append(loss)
            # Accuracy
            acc = np.mean((output > 0.5) == y) * 100
            self.accuracies.append(acc)
            # Backward
            d_output = output - y
            d_hidden = d_output.dot(self.W2.T) * hidden * (1 - hidden)
            # Update
            self.W2 -= 0.5 * hidden.T.dot(d_output)
            self.b2 -= 0.5 * np.sum(d_output)
            self.W1 -= 0.5 * X.T.dot(d_hidden)
            self.b1 -= 0.5 * np.sum(d_hidden, axis=0)
    def predict(self, X):
        hidden = self.sigmoid(np.dot(X, self.W1) + self.b1)
        return (self.sigmoid(np.dot(hidden, self.W2) + self.b2) > 0.5).astype(int).flatten()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler

# POST-LAB: Advanced NN with Momentum, L2, and Dropout on Iris Dataset
class AdvancedNN:
    def __init__(self, input_size, hidden_size, output_size, l2_lambda=0.01, dropout_rate=0.2):
        self.W1 = np.random.randn(input_size, hidden_size) * 0.1
        self.b1 = np.zeros((1, hidden_size))
        self.W2 = np.random.randn(hidden_size, output_size) * 0.1
        self.b2 = np.zeros((1, output_size))
        
        # Momentum parameters
        self.v_W1, self.v_b1 = 0, 0
        self.v_W2, self.v_b2 = 0, 0
        
        self.l2_lambda = l2_lambda
        self.dropout_rate = dropout_rate

    def sigmoid(self, x):
        return 1 / (1 + np.exp(-np.clip(x, -500, 500)))

    def softmax(self, x):
        exp_x = np.exp(x - np.max(x, axis=1, keepdims=True))
        return exp_x / np.sum(exp_x, axis=1, keepdims=True)

    def train(self, X, y, epochs=1000, lr=0.1, momentum=0.9):
        losses = []
        for i in range(epochs):
            # Forward Pass with Dropout
            hidden_input = np.dot(X, self.W1) + self.b1
            hidden_output = self.sigmoid(hidden_input)
            
            # Apply Dropout
            dropout_mask = (np.random.rand(*hidden_output.shape) > self.dropout_rate) / (1 - self.dropout_rate)
            hidden_output *= dropout_mask

            final_input = np.dot(hidden_output, self.W2) + self.b2
            output = self.softmax(final_input)

            # Loss calculation with L2 Regularization
            loss = -np.mean(np.sum(y * np.log(output + 1e-8), axis=1))
            l2_loss = (self.l2_lambda / 2) * (np.sum(self.W1**2) + np.sum(self.W2**2))
            losses.append(loss + l2_loss)

            # Backpropagation
            d_output = output - y
            d_W2 = np.dot(hidden_output.T, d_output) / X.shape[0] + self.l2_lambda * self.W2
            d_b2 = np.sum(d_output, axis=0, keepdims=True) / X.shape[0]
            
            d_hidden = np.dot(d_output, self.W2.T) * hidden_output * (1 - hidden_output) * dropout_mask
            d_W1 = np.dot(X.T, d_hidden) / X.shape[0] + self.l2_lambda * self.W1
            d_b1 = np.sum(d_hidden, axis=0, keepdims=True) / X.shape[0]

            # Update with Momentum
            self.v_W2 = momentum * self.v_W2 + lr * d_W2
            self.v_b2 = momentum * self.v_b2 + lr * d_b2
            self.v_W1 = momentum * self.v_W1 + lr * d_W1
            self.v_b1 = momentum * self.v_b1 + lr * d_b1

            self.W2 -= self.v_W2
            self.b2 -= self.v_b2
            self.W1 -= self.v_W1
            self.b1 -= self.v_b1

        return losses

    def predict(self, X):
        hidden = self.sigmoid(np.dot(X, self.W1) + self.b1)
        return np.argmax(self.softmax(np.dot(hidden, self.W2) + self.b2), axis=1)

# Iris Data Preparation
iris = load_iris()
scaler = StandardScaler()
X_scaled = scaler.fit_transform(iris.data)
encoder = OneHotEncoder(sparse_output=False)
y_encoded = encoder.fit_transform(iris.target.reshape(-1, 1))

X_train, X_test, y_train, y_test = train_test_split(X_scaled, y_encoded, test_size=0.2, random_state=42)

# Training
nn = AdvancedNN(input_size=4, hidden_size=8, output_size=3)
training_losses = nn.train(X_train, y_train, epochs=500, lr=0.5, momentum=0.9)

# Evaluation
preds = nn.predict(X_test)
true_labels = np.argmax(y_test, axis=1)
accuracy = np.mean(preds == true_labels) * 100

plt.plot(training_losses)
plt.title(f"Iris Dataset Training Loss (Test Acc: {accuracy:.2f}%)")
plt.xlabel("Epochs")
plt.ylabel("Loss")
plt.show()